# Controlled comparisons: explain a change in results
**Objectives:** separate representation choices, select only on validation, inspect actual neighbors, and compare errors with credible simple references and initialization variation.
**Prerequisites:** notebooks 01, 04 and 06; NumPy indexing and k-NN.
**Data:** the archived Antwerp anchor arrays reconstruct from the public sources. They preserve the fixed spatial partition, unlike notebook 01's chronological example. Numerical thread limits are fixed at two for the archive comparison; different numerical kernels can alter the resolution of tied or nearly tied neighbor distances.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
# Match the archived numerical execution context, including neighbor-search kernels.
numeric_threads = threadpool_limits(limits=2)
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd()/'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT/'data/manifest.json').exists()
sys.path.insert(0,str(ROOT/'scripts'))
REVISION=ROOT/'results/revision_v1'


## 1. Inspect the experimental unit and fixed partition
Rows are packet fingerprints; columns are gateways. Coordinates are targets. A model must not receive position as a predictor. Distances below convert the projected response back to latitude/longitude and use the geographic scoring function.

In [ ]:
from experiments import haversine
data=np.load(REVISION/'inputs/antwerp_anchor.npz',allow_pickle=False)
X,y=data['X'],data['y']
tr,va,te=[data['split']==p for p in ['train','validation','test']]
display(pd.Series({p:int((data['split']==p).sum()) for p in ['train','validation','test']}))
assert not (tr&va).any() and not (tr&te).any()
assert len(np.unique(data['row_id']))==len(X)
def geographic_error(truth,prediction):
    def ll(a):
        return 51.22+np.rad2deg(a[:,1]/6371000),4.40+np.rad2deg(a[:,0]/(6371000*np.cos(np.deg2rad(51.22))))
    return haversine(*ll(truth),*ll(prediction))
print('Features:',X.shape[1],'Missing entries:',np.isnan(X).sum())


## 2. Refit two complete pipelines
Both replace missing values with each gateway's **training mean** and standardize using training statistics. The only deliberate difference is appending an observation mask. The fallback for a gateway with no training reception is −140, a computational code.

For each representation, select k on validation before scoring test. This controls the data and selection rule, while allowing k to adapt to the changed geometry. Test scores are printed only after both selections.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
count=np.isfinite(X[tr]).sum(axis=0)
fill=np.divide(np.nansum(X[tr],axis=0),count,out=np.full(X.shape[1],-140.),where=count>0)
filled=np.where(np.isfinite(X),X,fill)
chosen={}; rows=[]; representations={}
for use_mask in [False,True]:
    raw=np.column_stack([filled,np.isfinite(X).astype(float)]) if use_mask else filled.copy()
    mean=raw[tr].mean(axis=0); scale=raw[tr].std(axis=0)
    scale[scale<1e-12]=1.
    z=(raw-mean)/scale; representations[use_mask]=z
    best=None
    for k in [3,9,21,45]:
        fitted=KNeighborsRegressor(n_neighbors=k,weights='distance',n_jobs=2).fit(z[tr],y[tr])
        score=geographic_error(y[va],fitted.predict(z[va])).mean()
        rows.append(dict(mask=use_mask,k=k,validation_mean_m=score))
        if best is None or score<best[0]:best=(score,k,fitted)
    chosen[use_mask]=best
display(pd.DataFrame(rows))
test_results=[]
for mask,(validation,k,model) in chosen.items():
    error=geographic_error(y[te],model.predict(representations[mask][te])).mean()
    test_results.append(dict(mask=mask,k=k,test_mean_m=error))
    saved=pd.read_csv(REVISION/'metrics.csv')
    expected=saved.loc[saved.case.eq(f'representation_mean_mask{int(mask)}_scale1'),'mae'].iloc[0]
    np.testing.assert_allclose(error,expected,rtol=1e-9)
display(pd.DataFrame(test_results))


## 3. Inspect geometry independently of the selected k
Hold nine neighbors fixed. Their overlap shows how much the representation changes local similarity. It does not tell us which missingness mechanism generated the data.

In [ ]:
from sklearn.neighbors import NearestNeighbors
neighbors=[]
for mask in [False,True]:
    z=representations[mask]
    neighbors.append(NearestNeighbors(n_neighbors=9,n_jobs=2).fit(z[tr]).kneighbors(z[te],return_distance=False))
overlap=np.array([len(set(a)&set(b))/9 for a,b in zip(*neighbors)])
print('Mean fraction of shared neighbors:',overlap.mean())
display(pd.read_csv(REVISION/'neighbor_overlap.csv'))


**Exercise:** Why does the original-scale sentinel experiment show almost no mask effect, while the standardized mean-imputation experiment changes substantially? Does the best of these eight test scores establish the best representation for a new city?

<details><summary>Feedback after your attempt</summary>

numerical weighting changes distance geometry. A 0/1 difference contributes little beside large RSSI differences; scaling can give it a different role. A filled mean also moves missing observations toward measured values. These are mechanisms of the representation, not evidence that absence means weak signal. Choosing among the eight representations by their test results is exploratory; a new confirmatory comparison needs another holdout.

</details>

## 4. Read the selection budget and reference errors
Include persistence and the historical mean when comparing forecasts. Show all declared initializations. Their range describes optimization sensitivity on the same test data; it is not a confidence interval over deployments.

In [ ]:
metrics=pd.read_csv(REVISION/'metrics.csv')
display(metrics[metrics.category.eq('anchor')][['case','model','seed','mae']])
selection=json.loads((REVISION/'selection.json').read_text())
display(pd.DataFrame([{ 'task_family':key, 'candidates':v['candidates'], 'selected':v['config']} for key,v in selection.items()]))


**Adaptation task:** define a new location tolerance. Specify a training-only representation and a validation budget, then explain whether an observed error difference changes the intended decision. Read the spatial-buffer controls before interpreting a low mean error as broad geographic coverage.